# Setup: MSA Benchmark Environment
Run this notebook once to install all tools and download the BAliBASE 3.0 dataset.

## 1. Install Python Dependencies

In [1]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *packages])

pip_install('biopython', 'tqdm')
print('Python dependencies installed.')

Python dependencies installed.


## 2. Install Bioinformatics Tools in WSL
This installs MAFFT, Clustal Omega, and the GNU `time` utility (needed for memory tracking).
MUSCLE5 is handled separately in the next cell because `apt` only provides the older MUSCLE3.

In [ ]:
import subprocess

def wsl(cmd, check=True, root=False):
    """Run a command in WSL. Pass root=True to run as root."""
    user_flag = ['-u', 'root'] if root else []
    # Always force non-interactive so apt/dpkg never shows interactive dialogs
    full_cmd  = f'DEBIAN_FRONTEND=noninteractive {cmd}' if root else cmd
    result = subprocess.run(
        ['wsl', *user_flag, '-e', 'bash', '-c', full_cmd],
        capture_output=True, text=True
    )
    if check and result.returncode != 0:
        print('STDERR:', result.stderr[:500])
        raise RuntimeError(f'WSL command failed (rc={result.returncode}): {cmd}')
    return result.stdout.strip()

# ── Clear stale locks and finish any interrupted dpkg configure ──────────
print('Clearing apt locks and repairing dpkg state...')
wsl(
    'rm -f /var/lib/apt/lists/lock '
    '       /var/lib/dpkg/lock '
    '       /var/lib/dpkg/lock-frontend '
    '       /var/cache/apt/archives/lock && '
    'dpkg --configure -a --force-confdef --force-confold',
    root=True, check=False
)

print('Installing mafft, clustalo, time...')
wsl(
    'apt-get update -qq --fix-missing && '
    'apt-get install -y --fix-broken mafft clustalo time',
    root=True
)
print('Done.')

## 3. Install MUSCLE5 (Linux binary for WSL)

Your Windows binary (`muscle-win64.v5.3.exe`) can't run inside WSL — we need the Linux build.
The cell below downloads the matching **v5.3** Linux binary directly into WSL.

In [3]:
MUSCLE5_URL = 'https://github.com/rcedgar/muscle/releases/download/v5.3/muscle-linux-x86.v5.3'

print('Downloading MUSCLE5 v5.3 Linux binary...')
wsl(f'wget -q -O /usr/local/bin/muscle5 "{MUSCLE5_URL}" && chmod +x /usr/local/bin/muscle5', root=True)

version = wsl('/usr/local/bin/muscle5 -version 2>&1', check=False)
print('MUSCLE5:', version or '(installed)')

wsl('ln -sf /usr/local/bin/muscle5 /usr/local/bin/muscle', root=True, check=False)
print('Done.')

MUSCLE5: muscle 5.3.linux64 [d9725ac]
Built Nov 10 2024 22:58:59
Done.


## 4. Install Additional Aligners: Kalign3, T-Coffee, FAMSA

Kalign3 and T-Coffee are available via `apt`. FAMSA requires downloading a pre-built binary from GitHub.

In [ ]:
# ── Kalign3 (fast, ~2 MB) ────────────────────────────────────────────────
print('Installing Kalign3...')
wsl('apt-get install -y kalign', root=True)
out = wsl('kalign --version 2>&1 | head -1', check=False)
print(f'  kalign: {out or "NOT FOUND"}')

# ── T-Coffee via conda (avoids the heavy apt dependency chain) ────────────
# apt t-coffee pulls in 300+ MB of dependencies and frequently stalls.
# We use the official pre-built Linux binary from the T-Coffee website instead.
TCOFFEE_URL = (
    'https://github.com/cbcrg/tcoffee/releases/download/Version_13.45.61.0/'
    'T-COFFEE_installer_Version_13.45.61.0_linux_x64.tar.gz'
)
print('\nDownloading T-Coffee binary (this may take ~30 s)...')
result = wsl(
    f'wget -q --timeout=60 -O /tmp/tcoffee.tar.gz "{TCOFFEE_URL}" 2>&1 && '
    f'mkdir -p /usr/local/tcoffee && '
    f'tar -xzf /tmp/tcoffee.tar.gz -C /usr/local/tcoffee --strip-components=1 2>&1 && '
    f'ln -sf /usr/local/tcoffee/bin/t_coffee /usr/local/bin/t_coffee',
    root=True, check=False
)
out = wsl('t_coffee -version 2>&1 | head -2', check=False)
print(f'  t_coffee: {out.strip() or "NOT FOUND — see note below"}')
if not out.strip():
    print('  NOTE: T-Coffee binary install failed. You can skip it;')
    print('        remove "t_coffee" from ALIGNERS in 02_benchmark.ipynb.')

# ── FAMSA — single pre-built binary (~5 MB, very fast) ───────────────────
FAMSA_URL = 'https://github.com/refresh-bio/FAMSA/releases/download/v2.2.2/famsa-linux-x86_64'
print('\nDownloading FAMSA v2.2.2...')
wsl(
    f'wget -q --timeout=60 -O /usr/local/bin/famsa "{FAMSA_URL}" && '
    f'chmod +x /usr/local/bin/famsa',
    root=True, check=False
)
out = wsl('famsa --version 2>&1 | head -1', check=False)
print(f'  famsa: {out or "NOT FOUND"}')

print('\nDone.')

## 4. Extract BAliBASE 3.0 Archives

The 4 archives (`BAliBASE_R1-5.tar_.gz`, `R6-8`, `R9`, `R10`) are already in the project root.
This cell extracts them all into `data/bb3_release/`.

In [4]:
from pathlib import Path

PROJECT_DIR = Path(r'C:\Users\anand\Desktop\SEM 4\CS 502\Project')
DATA_DIR    = PROJECT_DIR / 'data'

def to_wsl_path(win_path):
    p = str(win_path).replace('\\', '/')
    if len(p) >= 2 and p[1] == ':':
        p = f'/mnt/{p[0].lower()}{p[2:]}'
    return p

archives = sorted(PROJECT_DIR.glob('BAliBASE_R*.tar_.gz'))
if not archives:
    print('No BAliBASE archives found in project root.')
else:
    for archive in archives:
        print(f'Extracting {archive.name}...')
        wsl(f'tar -xzf "{to_wsl_path(archive)}" -C "{to_wsl_path(DATA_DIR)}"', root=True)
    print('Done.')

Extracting BAliBASE_R1-5.tar_.gz...
Extracting BAliBASE_R10.tar_.gz...
Extracting BAliBASE_R6-8.tar_.gz...
Extracting BAliBASE_R9.tar_.gz...
Done.


## 5. Verify Setup

In [ ]:
import os
from pathlib import Path
from Bio import AlignIO  # noqa: F401 — just checking import works

PROJECT_DIR = Path(r'C:\Users\anand\Desktop\SEM 4\CS 502\Project')
BB3_DIR     = PROJECT_DIR / 'data' / 'bb3_release'
RV_SETS     = ['RV11', 'RV12', 'RV20', 'RV30', 'RV40', 'RV50']

print('=== Tool versions ===')
for tool, cmd in [
    ('MAFFT',      'mafft --version 2>&1 | head -1'),
    ('MUSCLE5',    'muscle5 --version 2>&1 | head -1'),
    ('Clustal O',  'clustalo --version 2>&1 | head -1'),
    ('/usr/bin/time', '/usr/bin/time --version 2>&1 | head -1'),
]:
    out = wsl(cmd, check=False)
    print(f'  {tool:12s}: {out or "NOT FOUND"}')

print()
print('=== BAliBASE data ===')
total_problems = 0
for rv in RV_SETS:
    rv_dir = BB3_DIR / rv
    if rv_dir.exists():
        n = len(list(rv_dir.glob('*.tfa')))
        total_problems += n
        print(f'  {rv}: {n} problems')
    else:
        print(f'  {rv}: MISSING')

print(f'\nTotal: {total_problems} alignment problems')
print(f'Expected alignments: {total_problems * 4} (4 aligners)')

if total_problems > 0:
    print('\nSetup complete — proceed to 02_benchmark.ipynb')
else:
    print('\nWARNING: No BAliBASE data found. Check the download step above.')

=== Tool versions ===


c:\Users\anand\AppData\Local\Programs\Python\Python311\Lib\site-packages\Bio\__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: c:\Users\anand\AppData\Local\Programs\Python\Python311\Lib\site-packages
  warnings.warn(


  MAFFT       : v7.505 (2022/Apr/10)
  MUSCLE5     : muscle 5.3.linux64 [d9725ac]
  Clustal O   : 1.2.4
  /usr/bin/time: time (GNU Time) UNKNOWN

=== BAliBASE data ===
  RV11: 76 problems
  RV12: 88 problems
  RV20: 82 problems
  RV30: 60 problems
  RV40: 49 problems
  RV50: 31 problems

Total: 386 alignment problems
Expected alignments: 1544 (4 aligners)

Setup complete — proceed to 02_benchmark.ipynb


: 